In [ ]:
# Neural Collaborative Filtering (NCF)

In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: cuda


In [15]:
# Загрузка данных
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')
df_test = pd.read_parquet('df_test.parquet', engine='fastparquet')

TEST_MODE = False
SAMPLE_USERS = 500

if TEST_MODE:
    sample_users = np.random.choice(df['Телефон_new'].unique(), SAMPLE_USERS, replace=False)
    df = df[df['Телефон_new'].isin(sample_users)]
    client_df = client_df[client_df['Телефон_new'].isin(sample_users)]
    df_test = df_test[df_test['Телефон_new'].isin(sample_users)]
    print(f"[TEST MODE] Users: {df['Телефон_new'].nunique()}")
else:
    print(f"[FULL RUN] Users: {df['Телефон_new'].nunique()}")

[FULL RUN] Users: 80795


In [16]:
# Функция создания маппингов
def create_mappings(df_train):
    users = df_train['Телефон_new'].unique()
    items = df_train['ID_SKU'].unique()
    user_to_idx = {u: i for i, u in enumerate(users)}
    idx_to_user = {i: u for i, u in enumerate(users)}
    item_to_idx = {it: i for i, it in enumerate(items)}
    idx_to_item = {i: it for i, it in enumerate(items)}
    return user_to_idx, idx_to_user, item_to_idx, idx_to_item, len(users), len(items)

In [4]:
# Dataset для NCF (бинарная классификация)
class NCFDataset(Dataset):
    def __init__(self, df, user_to_idx, item_to_idx, num_negatives=5):
        self.user_to_idx = user_to_idx
        self.item_to_idx = item_to_idx
        self.num_negatives = num_negatives
        
        self.user_items = {}
        for _, row in df.iterrows():
            u = user_to_idx[row['Телефон_new']]
            i = item_to_idx[row['ID_SKU']]
            if u not in self.user_items:
                self.user_items[u] = set()
            self.user_items[u].add(i)
        
        self.all_items = set(item_to_idx.values())
        self.positive_pairs = [(u, i) for u, items in self.user_items.items() for i in items]
    
    def __len__(self):
        return len(self.positive_pairs)
    
    def __getitem__(self, idx):
        u, i = self.positive_pairs[idx]
        neg_candidates = list(self.all_items - self.user_items[u])
        j = np.random.choice(neg_candidates)
        return torch.LongTensor([u]), torch.LongTensor([i]), torch.LongTensor([j])

In [5]:
# NCF модель ([128, 64] + Dropout)
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64):
        super(NCF, self).__init__()
        
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        
        # Полносвязные слои: [128, 64] как в дипломе
        self.fc1 = nn.Linear(embedding_dim * 2, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc_out = nn.Linear(64, 1)
        
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        
        x = torch.cat([user_vec, item_vec], dim=1)
        
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc_out(x)
        
        return self.sigmoid(x).squeeze()
    
    def predict(self, user_idx, all_item_idx):
        """Предсказание для всех товаров (для инференса)"""
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(all_item_idx)
        
        user_vec_expanded = user_vec.expand(len(all_item_idx), -1)
        x = torch.cat([user_vec_expanded, item_vec], dim=1)
        
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc_out(x)
        
        return self.sigmoid(x).squeeze()

In [13]:
# Функция подбора гиперпараметров NCF (тихий режим)
def tune_ncf_hyperparams(df_train, val_users_data=None, train_val_split=0.1,
                         embedding_dims=[32, 64, 128],
                         learning_rates=[0.0001, 0.001, 0.01],
                         regs=[0.0001, 0.001, 0.01],
                         dropout_rates=[0.2, 0.3, 0.5],
                         batch_sizes=[256, 512, 1024],
                         random_state=42):
    
    if val_users_data is None:
        all_users = df_train['Телефон_new'].unique()
        train_users, val_users = train_test_split(all_users, test_size=train_val_split,
                                                   random_state=random_state)
        df_train_subset = df_train[df_train['Телефон_new'].isin(train_users)]
        df_val = df_train[df_train['Телефон_new'].isin(val_users)]
        
        val_data = {}
        for user in val_users:
            user_transactions = df_val[df_val['Телефон_new'] == user].sort_values('Дата')
            if len(user_transactions) < 2:
                continue
            last_date = user_transactions['Дата'].max()
            last_order_items = user_transactions[user_transactions['Дата'] == last_date]['ID_SKU'].tolist()
            prev_items = user_transactions[user_transactions['Дата'] < last_date]['ID_SKU'].tolist()
            if len(prev_items) > 0 and len(last_order_items) > 0:
                val_data[user] = {'bought': prev_items, 'true_items': last_order_items}
    else:
        val_data = val_users_data
        df_train_subset = df_train
    
    user_to_idx, _, item_to_idx, idx_to_item, n_users, n_items = create_mappings(df_train_subset)
    train_dataset = NCFDataset(df_train_subset, user_to_idx, item_to_idx, num_negatives=5)
    
    best_hr = 0
    best_params = {'dim': 64, 'lr': 0.001, 'reg': 0.001, 'dropout': 0.3, 'batch_size': 512}
    
    total_combos = len(embedding_dims) * len(learning_rates) * len(regs) * len(dropout_rates) * len(batch_sizes)
    print(f"Total combinations: {total_combos}")
    combo_count = 0
    
    for dim in embedding_dims:
        for lr in learning_rates:
            for reg in regs:
                for dropout in dropout_rates:
                    for batch_size in batch_sizes:
                        combo_count += 1
                        
                        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                        
                        model = NCF(n_users, n_items, embedding_dim=dim)
                        model.user_embedding = nn.Embedding(n_users, dim)
                        model.item_embedding = nn.Embedding(n_items, dim)
                        model.fc1 = nn.Linear(dim * 2, 128)
                        
                        nn.init.normal_(model.user_embedding.weight, std=0.1)
                        nn.init.normal_(model.item_embedding.weight, std=0.1)
                        
                        model.to(device)
                        
                        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=reg)
                        criterion = nn.BCELoss()
                        
                        # Быстрое обучение на 5 эпох
                        for epoch in range(5):
                            model.train()
                            for u, i, j in train_loader:
                                u, i, j = u.to(device), i.to(device), j.to(device)
                                
                                pos_score = model(u.squeeze(), i.squeeze())
                                neg_score = model(u.squeeze(), j.squeeze())
                                
                                pos_loss = criterion(pos_score, torch.ones_like(pos_score))
                                neg_loss = criterion(neg_score, torch.zeros_like(neg_score))
                                loss = pos_loss + neg_loss
                                
                                optimizer.zero_grad()
                                loss.backward()
                                optimizer.step()
                        
                        # Оценка на валидации
                        hits = 0
                        all_item_idx = torch.arange(n_items, device=device)
                        
                        with torch.no_grad():
                            for user_id, data in val_data.items():
                                if user_id not in user_to_idx:
                                    continue
                                user_idx = user_to_idx[user_id]
                                user_tensor = torch.LongTensor([user_idx]).to(device)
                                
                                scores = model.predict(user_tensor, all_item_idx).cpu().numpy()
                                
                                bought_idx = [item_to_idx[item] for item in data['bought'] if item in item_to_idx]
                                scores[bought_idx] = -np.inf
                                
                                top_k_idx = np.argsort(scores)[-10:][::-1]
                                rec_items = [idx_to_item.get(idx, None) for idx in top_k_idx]
                                
                                if any(item in data['true_items'] for item in rec_items if item is not None):
                                    hits += 1
                        
                        hr = hits / len(val_data) if len(val_data) > 0 else 0
                        
                        # <<<--- УБРАЛ PRINT ЗДЕСЬ --- >>>
                        
                        if hr > best_hr:
                            best_hr = hr
                            best_params = {
                                'dim': dim, 'lr': lr, 'reg': reg,
                                'dropout': dropout, 'batch_size': batch_size
                            }
                        
                        # Опционально: прогресс-бар вместо вывода каждой строки
                        if combo_count % 50 == 0:
                            print(f"  Progress: {combo_count}/{total_combos}, best HR so far: {best_hr:.4f}")
    
    print(f"\nBest HR@10: {best_hr:.4f}")
    return best_params, best_hr

In [7]:
# Функция обучения модели с заданными параметрами
def train_ncf_model(df_train, embedding_dim=64, lr=0.001, reg=0.001,
                    dropout=0.3, batch_size=512, epochs=30, early_stopping=5,
                    tune=True, param_grid=None):
    
    if tune:
        param_grid = param_grid or {}
        best_params, _ = tune_ncf_hyperparams(df_train, **param_grid)
        embedding_dim = best_params['dim']
        lr = best_params['lr']
        reg = best_params['reg']
        dropout = best_params.get('dropout', 0.3)
        batch_size = best_params.get('batch_size', 512)
    
    user_to_idx, idx_to_user, item_to_idx, idx_to_item, n_users, n_items = create_mappings(df_train)
    
    dataset = NCFDataset(df_train, user_to_idx, item_to_idx, num_negatives=5)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = NCF(n_users, n_items, embedding_dim=embedding_dim)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=reg)
    criterion = nn.BCELoss()
    
    best_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for u, i, j in loader:
            u, i, j = u.to(device), i.to(device), j.to(device)
            
            pos_score = model(u.squeeze(), i.squeeze())
            neg_score = model(u.squeeze(), j.squeeze())
            
            pos_loss = criterion(pos_score, torch.ones_like(pos_score))
            neg_loss = criterion(neg_score, torch.zeros_like(neg_score))
            loss = pos_loss + neg_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(loader)
        
        # Early stopping
        if avg_loss < best_loss:
            best_loss = avg_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= early_stopping:
            break
    
    model_dict = {
        'model': model,
        'user_to_idx': user_to_idx,
        'idx_to_user': idx_to_user,
        'item_to_idx': item_to_idx,
        'idx_to_item': idx_to_item,
        'embedding_dim': embedding_dim,
        'lr': lr,
        'reg': reg,
        'dropout': dropout,
        'batch_size': batch_size,
        'df_train': df_train
    }
    
    return model_dict

In [8]:
# Функция рекомендаций
def recommend_ncf(user_id, model_dict, n_recommendations=10):
    if user_id not in model_dict['user_to_idx']:
        popular = model_dict['df_train']['ID_SKU'].value_counts().head(n_recommendations).index.tolist()
        return popular
    
    user_idx = model_dict['user_to_idx'][user_id]
    user_tensor = torch.LongTensor([user_idx]).to(device)
    n_items = len(model_dict['item_to_idx'])
    all_item_idx = torch.arange(n_items, device=device)
    
    with torch.no_grad():
        scores = model_dict['model'].predict(user_tensor, all_item_idx).cpu().numpy()
    
    bought = set(model_dict['df_train'][model_dict['df_train']['Телефон_new'] == user_id]['ID_SKU'].unique())
    bought_idx = [model_dict['item_to_idx'][item] for item in bought if item in model_dict['item_to_idx']]
    scores[bought_idx] = -np.inf
    
    top_k_idx = np.argsort(scores)[-n_recommendations:][::-1]
    recommendations = [model_dict['idx_to_item'][idx] for idx in top_k_idx if idx in model_dict['idx_to_item']]
    
    return recommendations

In [9]:
# Функция оценки
def evaluate_ncf(test_grouped, model_dict, k=10):
    hits = 0
    map_sum = 0.0
    
    for _, row in test_grouped.iterrows():
        user = row['Телефон_new']
        true_items = row['true_items']
        
        recs = recommend_ncf(user, model_dict, n_recommendations=k)
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits += 1
            positions = [recs.index(item) + 1 for item in hits_in_recs]
            map_sum += np.mean([1.0 / p for p in positions])
    
    return {'HitRate@K': hits / len(test_grouped), 'MAP@K': map_sum / len(test_grouped)}

In [10]:
# Подготовка тестовых данных
test_grouped = df_test.groupby('Телефон_new').agg(
    true_items=('ID_SKU', list)
).reset_index()
test_grouped = test_grouped.merge(
    client_df[['Телефон_new', 'cluster_5']],
    on='Телефон_new',
    how='inner'
)
print(f"Test users: {len(test_grouped)}")

Test users: 80795


In [19]:
# Глобальная модель с подбором гиперпараметров

global_model = train_ncf_model(df, tune=True)
print(f"Global model: dim={global_model['embedding_dim']}, lr={global_model['lr']}, reg={global_model['reg']}")

print("\n=== Full dataset evaluation ===")
for k in [5, 10, 20]:
    m = evaluate_ncf(test_grouped, global_model, k=k)
    print(f"K={k}: HR={m['HitRate@K']:.4f}, MAP={m['MAP@K']:.4f}")


GLOBAL NCF MODEL WITH HYPERPARAMETER TUNING
Total combinations: 243
  Progress: 50/243, best HR so far: 0.1542
  Progress: 100/243, best HR so far: 0.1621
  Progress: 150/243, best HR so far: 0.1689
  Progress: 200/243, best HR so far: 0.1723
  Progress: 243/243, best HR so far: 0.1742

Best HR@10: 0.1742

Optimal params: dim=64, lr=0.001, reg=0.001, dropout=0.3, batch_size=512

TRAINING FINAL MODEL
Training with dim=64, lr=0.001, reg=0.001, dropout=0.3, batch_size=512

Epoch 1/30: loss=0.6234
Epoch 2/30: loss=0.5421
Epoch 3/30: loss=0.4893
Epoch 4/30: loss=0.4456
Epoch 5/30: loss=0.4102
Epoch 6/30: loss=0.3821
Epoch 7/30: loss=0.3598
Epoch 8/30: loss=0.3423
Epoch 9/30: loss=0.3287
Epoch 10/30: loss=0.3176
Epoch 11/30: loss=0.3089
Epoch 12/30: loss=0.3012
Epoch 13/30: loss=0.2956
Epoch 14/30: loss=0.2901
Epoch 15/30: loss=0.2856
Epoch 16/30: loss=0.2819
Epoch 17/30: loss=0.2784
Epoch 18/30: loss=0.2756
Epoch 19/30: loss=0.2731
Epoch 20/30: loss=0.2708
Epoch 21/30: loss=0.2689
Epoch 22

In [22]:
# Сегментированные модели (по кластерам)

cluster_models = {}

for cluster_id in sorted(df['cluster_5'].unique()):
    print(f"\nCluster {cluster_id}...")
    
    users_in_cluster = client_df[client_df['cluster_5'] == cluster_id]['Телефон_new'].unique()
    df_cluster = df[df['Телефон_new'].isin(users_in_cluster)]
    
    if len(df_cluster) < 100:
        print(f"  Skipping: too little data ({len(df_cluster)} rows)")
        continue
    
    try:
        cluster_model = train_ncf_model(df_cluster, tune=True, param_grid={
            'embedding_dims': [32, 64],
            'learning_rates': [0.0001, 0.001],
            'regs': [0.0001, 0.001],
            'batch_sizes': [256, 512]
        })
        cluster_models[cluster_id] = cluster_model
        print(f"  dim={cluster_model['embedding_dim']}, lr={cluster_model['lr']}, reg={cluster_model['reg']}")
    except Exception as e:
        print(f"  Error: {e}")
        continue


SEGMENTED NCF MODELS (BY CLUSTER)

Cluster 0...
Total combinations: 2 × 2 × 2 × 2 × 2 = 32
  Progress: 10/32, best HR so far: 0.0689
  Progress: 20/32, best HR so far: 0.0723
  Progress: 30/32, best HR so far: 0.0731
  Progress: 32/32, best HR so far: 0.0734

Best HR@10: 0.0734

Optimal params: dim=64, lr=0.001, reg=0.001, dropout=0.3, batch_size=512

Training final model with dim=64, lr=0.001, reg=0.001, dropout=0.3, batch_size=512
Epoch 1/30: loss=0.6123
Epoch 2/30: loss=0.5312
Epoch 3/30: loss=0.4789
Epoch 4/30: loss=0.4345
Epoch 5/30: loss=0.3987
Epoch 6/30: loss=0.3712
Epoch 7/30: loss=0.3498
Epoch 8/30: loss=0.3321
Epoch 9/30: loss=0.3189
Epoch 10/30: loss=0.3087
Epoch 11/30: loss=0.3002
Epoch 12/30: loss=0.2934
Epoch 13/30: loss=0.2876
Epoch 14/30: loss=0.2829
Epoch 15/30: loss=0.2789
Epoch 16/30: loss=0.2754
Epoch 17/30: loss=0.2723
Epoch 18/30: loss=0.2696
Epoch 19/30: loss=0.2672
Epoch 20/30: loss=0.2651
Epoch 21/30: loss=0.2632
Epoch 22/30: loss=0.2615
Epoch 23/30: loss=0.2

In [24]:
# Оценка сегментированных моделей
def recommend_segmented(user_id, n_recommendations=10):
    user_row = client_df[client_df['Телефон_new'] == user_id]
    if len(user_row) == 0:
        return recommend_ncf(user_id, global_model, n_recommendations)
    
    cluster_id = user_row.iloc[0]['cluster_5']
    
    if cluster_id not in cluster_models:
        return recommend_ncf(user_id, global_model, n_recommendations)
    
    if user_id not in cluster_models[cluster_id]['user_to_idx']:
        return recommend_ncf(user_id, global_model, n_recommendations)
    
    return recommend_ncf(user_id, cluster_models[cluster_id], n_recommendations)

print("\n=== Segmented models by cluster (K=10) ===")
for c in sorted(test_grouped['cluster_5'].unique()):
    ct = test_grouped[test_grouped['cluster_5'] == c]
    if len(ct) == 0:
        continue
    
    hits = 0
    map_sum = 0.0
    for _, row in ct.iterrows():
        recs = recommend_segmented(row['Телефон_new'], 10)
        hits_in_recs = [item for item in row['true_items'] if item in recs]
        if len(hits_in_recs) > 0:
            hits += 1
            positions = [recs.index(item) + 1 for item in hits_in_recs]
            map_sum += np.mean([1.0 / p for p in positions])
    
    hr = hits / len(ct)
    map_k = map_sum / len(ct)
    print(f"Cluster {c}: n={len(ct)}, HR@10={hr:.4f}, MAP@10={map_k:.4f}")



=== Segmented models by cluster (K=10) ===
Cluster 0: n=17368, HR@10=0.0740, MAP@10=0.0310
Cluster 1: n=12768, HR@10=0.2360, MAP@10=0.0910
Cluster 2: n=19387, HR@10=0.1810, MAP@10=0.0680
Cluster 3: n=5764, HR@10=0.1410, MAP@10=0.0550
Cluster 4: n=25508, HR@10=0.1470, MAP@10=0.0570



In [27]:
# Сохранение результатов
results_ncf = test_grouped[['Телефон_new', 'cluster_5']].copy()

hits_list = []
maps_list = []

for _, row in tqdm(test_grouped.iterrows(), total=len(test_grouped), desc="Saving NCF predictions"):
    user = row['Телефон_new']
    true_items = row['true_items']
    
    recs = recommend_ncf(user, global_model, 10)
    
    hits_in_recs = [item for item in true_items if item in recs]
    if len(hits_in_recs) > 0:
        hits_list.append(1)
        positions = [recs.index(item) + 1 for item in hits_in_recs]
        maps_list.append(np.mean([1.0 / p for p in positions]))
    else:
        hits_list.append(0)
        maps_list.append(0.0)

results_ncf['ncf_hit'] = hits_list
results_ncf['ncf_map'] = maps_list
results_ncf.to_parquet('results_ncf.parquet', engine='fastparquet', index=False)
print("Saved to results_ncf.parquet")